# Comparative Analysis of GAN Loss Functions
## Visualization Notebook — 5 Loss Functions

**Author:** Sanskar Kushwah | NIT Srinagar  
**Datasets:** CIFAR-10 · EuroSAT · CheXpert  
**Models:** Standard GAN · LSGAN · WGAN · WGAN-GP · Hinge Loss

---
| Cell | Plot | Description |
|------|------|-------------|
| 2 | Setup | Imports, palette, helpers |
| 3 | Load Data | Read CSV files |
| 4 | Plot 01 | FID curves over epochs — CIFAR-10 |
| 5 | Plot 02 | FID curves over epochs — EuroSAT |
| 6 | Plot 03 | G & D Loss curves (5 subplots) — CIFAR-10 |
| 7 | Plot 04 | G & D Loss curves (5 subplots) — EuroSAT |
| 8 | Plot 05 | Mode Variance over epochs — CIFAR-10 |
| 9 | Plot 06 | Mode Variance over epochs — EuroSAT |
| 10 | Plot 07 | Best FID bar chart — CIFAR-10 & EuroSAT |
| 11 | Plot 08 | Mode Variance bar — CIFAR-10 vs EuroSAT |
| 12 | Plot 09 | Quality–Diversity scatter (FID vs ModeVar) |
| 13 | Run All | Regenerate all plots at once |

> **CSV files required** (put them in a `csv/` folder):
> - `csv/cifer_all_6.csv`
> - `csv/euro_5229_graph.csv`

In [ ]:
# Cell 1 — Install dependencies (run once)
import subprocess, sys
for p in ["numpy", "pandas", "matplotlib", "scipy"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", p, "-q"])
print("All packages ready.")

In [ ]:
# Cell 2 — Setup: imports, palette, helpers
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.ndimage import uniform_filter1d

warnings.filterwarnings("ignore")
os.makedirs("plots", exist_ok=True)

# ── Global plot style ─────────────────────────────────────────────────────────
matplotlib.rcParams.update({
    "font.family":        "DejaVu Sans",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.alpha":         0.22,
    "grid.linestyle":     "--",
    "grid.linewidth":     0.6,
    "figure.dpi":         120,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.15,
    "axes.titlesize":     13,
    "axes.labelsize":     11,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
})

# ── Color palette — 5 models only ────────────────────────────────────────────
PAL = {
    "STANDARD": "#E24B4A",   # red
    "LSGAN":    "#EF9F27",   # amber
    "WGAN":     "#378ADD",   # blue
    "WGANGP":   "#1D9E75",   # teal/green
    "HINGE":    "#7F77DD",   # purple
}
LABELS = {
    "STANDARD": "Standard GAN",
    "LSGAN":    "LSGAN",
    "WGAN":     "WGAN",
    "WGANGP":   "WGAN-GP",
    "HINGE":    "Hinge Loss",
}
ORDER = list(PAL.keys())

def col(key):  return PAL[key]
def lbl(key):  return LABELS[key]

def smooth(arr, w=5):
    arr = np.array(arr, dtype=float)
    return uniform_filter1d(arr, size=min(w, len(arr)))

def save_fig(fig, name):
    path = f"plots/{name}.png"
    fig.savefig(path)
    print(f"  saved → {path}")

print("Setup complete.")

In [ ]:
# Cell 3 — Load CSV data
CIFAR   = pd.read_csv("csv/cifer_all_6.csv")
EURO150 = pd.read_csv("csv/euro_5229_graph.csv")

# Keep only the 5 main loss functions
CIFAR   = CIFAR[CIFAR["Experiment"].isin(ORDER)].reset_index(drop=True)
EURO150 = EURO150[EURO150["Experiment"].isin(ORDER)].reset_index(drop=True)

def get_col(df, exp, col_name):
    return df[df["Experiment"] == exp][col_name].tolist()

def get_fid(df, exp):
    sub = df[(df["Experiment"] == exp) & df["FID"].notna()]
    return sub["Epoch"].tolist(), sub["FID"].tolist()

print("CIFAR-10  experiments:", CIFAR["Experiment"].unique().tolist())
print("EuroSAT   experiments:", EURO150["Experiment"].unique().tolist())

In [ ]:
# Plot 01 — FID curves over epochs · CIFAR-10
fig, ax = plt.subplots(figsize=(11, 5.5))

for key in ORDER:
    ep, fid = get_fid(CIFAR, key)
    ax.plot(ep, fid, color=col(key), lw=1.8,
            marker="o", markersize=4, label=lbl(key))

ax.set_xlabel("Epoch")
ax.set_ylabel("FID score  (↓ lower is better)")
ax.set_title("FID Score over Training Epochs — CIFAR-10")
ax.legend(fontsize=10, ncol=2, framealpha=0.85)
plt.tight_layout()
plt.show()
save_fig(fig, "01_fid_curves_cifar10")

In [ ]:
# Plot 02 — FID curves over epochs · EuroSAT
fig, ax = plt.subplots(figsize=(11, 5.5))

for key in ORDER:
    ep, fid = get_fid(EURO150, key)
    ax.plot(ep, fid, color=col(key), lw=1.8,
            marker="o", markersize=4, label=lbl(key))

ax.set_xlabel("Epoch")
ax.set_ylabel("FID score  (↓ lower is better)")
ax.set_title("FID Score over Training Epochs — EuroSAT")
ax.legend(fontsize=10, ncol=2, framealpha=0.85)
plt.tight_layout()
plt.show()
save_fig(fig, "02_fid_curves_eurosat")

In [ ]:
# Plot 03 — Generator & Discriminator Loss Curves · CIFAR-10
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, key in zip(axes[:5], ORDER):
    ep   = CIFAR[CIFAR["Experiment"] == key]["Epoch"].tolist()
    gl_s = smooth(get_col(CIFAR, key, "G_Loss"))
    dl_s = smooth(get_col(CIFAR, key, "D_Loss"))

    ax.plot(ep, gl_s, color=col(key), lw=2.0, label="Generator")
    ax.plot(ep, dl_s, color=col(key), lw=1.4, ls="--", alpha=0.6,
            label="Discriminator")
    ax.set_title(lbl(key), color=col(key), fontweight="bold")
    ax.set_xlabel("Epoch", fontsize=9)
    ax.set_ylabel("Loss",  fontsize=9)
    ax.legend(fontsize=8, framealpha=0.7)

# Hide the 6th (empty) subplot
axes[5].set_visible(False)

fig.suptitle("Generator & Discriminator Loss Curves — CIFAR-10\n"
             "Solid = Generator    Dashed = Discriminator",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()
save_fig(fig, "03_loss_curves_cifar10")

In [ ]:
# Plot 04 — Generator & Discriminator Loss Curves · EuroSAT
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, key in zip(axes[:5], ORDER):
    sub  = EURO150[EURO150["Experiment"] == key]
    ep   = sub["Epoch"].tolist()
    gl_s = smooth(sub["G_Loss"].tolist())
    dl_s = smooth(sub["D_Loss"].tolist())

    ax.plot(ep, gl_s, color=col(key), lw=2.0, label="Generator")
    ax.plot(ep, dl_s, color=col(key), lw=1.4, ls="--", alpha=0.6,
            label="Discriminator")
    ax.set_title(lbl(key), color=col(key), fontweight="bold")
    ax.set_xlabel("Epoch", fontsize=9)
    ax.set_ylabel("Loss",  fontsize=9)
    ax.legend(fontsize=8, framealpha=0.7)

axes[5].set_visible(False)

fig.suptitle("Generator & Discriminator Loss Curves — EuroSAT\n"
             "Solid = Generator    Dashed = Discriminator",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()
save_fig(fig, "04_loss_curves_eurosat")

In [ ]:
# Plot 05 — Mode Variance over epochs · CIFAR-10
fig, ax = plt.subplots(figsize=(11, 5.5))

for key in ORDER:
    ep = CIFAR[CIFAR["Experiment"] == key]["Epoch"].tolist()
    mv = smooth(get_col(CIFAR, key, "ModeVar"), w=7)
    ax.plot(ep, mv, color=col(key), lw=1.8, label=lbl(key))

ax.set_xlabel("Epoch")
ax.set_ylabel("Mode Variance  (↑ higher = more diverse)")
ax.set_title("Mode Variance over Training — CIFAR-10")
ax.legend(fontsize=10, ncol=2, framealpha=0.85)
plt.tight_layout()
plt.show()
save_fig(fig, "05_modevar_curves_cifar10")

In [ ]:
# Plot 06 — Mode Variance over epochs · EuroSAT
fig, ax = plt.subplots(figsize=(11, 5.5))

for key in ORDER:
    sub = EURO150[EURO150["Experiment"] == key]
    ep  = sub["Epoch"].tolist()
    mv  = smooth(sub["ModeVar"].tolist(), w=7)
    ax.plot(ep, mv, color=col(key), lw=1.8, label=lbl(key))

ax.set_xlabel("Epoch")
ax.set_ylabel("Mode Variance  (↑ higher = more diverse)")
ax.set_title("Mode Variance over Training — EuroSAT")
ax.legend(fontsize=10, ncol=2, framealpha=0.85)
plt.tight_layout()
plt.show()
save_fig(fig, "06_modevar_curves_eurosat")

In [ ]:
# Plot 07 — Best FID bar chart · CIFAR-10 & EuroSAT
best_cifar = {k: min(get_fid(CIFAR,   k)[1]) for k in ORDER}
best_euro  = {k: min(get_fid(EURO150, k)[1]) for k in ORDER}

x = np.arange(len(ORDER))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 6))

bars_c = ax.bar(x - w/2, [best_cifar[k] for k in ORDER], w,
                color=[col(k) for k in ORDER],
                edgecolor="white", zorder=3, label="CIFAR-10")
bars_e = ax.bar(x + w/2, [best_euro[k]  for k in ORDER], w,
                color=[col(k) for k in ORDER],
                edgecolor="white", alpha=0.55, hatch="///", zorder=3,
                label="EuroSAT")

for bar in list(bars_c) + list(bars_e):
    v = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.8,
            f"{v:.1f}", ha="center", va="bottom", fontsize=8, color="#444")

ax.set_xticks(x)
ax.set_xticklabels([LABELS[k] for k in ORDER], rotation=15, ha="right")
ax.set_ylabel("Best FID  (↓ lower is better)")
ax.set_title("Best FID — CIFAR-10 vs EuroSAT")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()
save_fig(fig, "07_best_fid_comparison")

In [ ]:
# Plot 08 — Final Mode Variance bar · CIFAR-10 vs EuroSAT
mc = {k: CIFAR[CIFAR["Experiment"]   == k]["ModeVar"].iloc[-1] for k in ORDER}
me = {k: EURO150[EURO150["Experiment"] == k]["ModeVar"].iloc[-1] for k in ORDER}

x = np.arange(len(ORDER))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 5.5))

bars_c = ax.bar(x - w/2, [mc[k] for k in ORDER], w,
                color=[col(k) for k in ORDER],
                edgecolor="white", zorder=3, label="CIFAR-10")
bars_e = ax.bar(x + w/2, [me[k] for k in ORDER], w,
                color=[col(k) for k in ORDER],
                edgecolor="white", alpha=0.55, hatch="///", zorder=3,
                label="EuroSAT")

for bar in list(bars_c) + list(bars_e):
    v = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.002,
            f"{v:.3f}", ha="center", va="bottom", fontsize=7.5, color="#444")

ax.set_xticks(x)
ax.set_xticklabels([LABELS[k] for k in ORDER], rotation=15, ha="right")
ax.set_ylabel("Mode Variance  (↑ higher = more diverse)")
ax.set_title("Final Mode Variance — CIFAR-10 vs EuroSAT")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()
save_fig(fig, "08_modevar_bar_comparison")

In [ ]:
# Plot 09 — Quality–Diversity Scatter (FID vs ModeVar) · CIFAR-10
bf = {k: min(get_fid(CIFAR, k)[1]) for k in ORDER}
fm = {k: CIFAR[CIFAR["Experiment"] == k]["ModeVar"].iloc[-1] for k in ORDER}

fig, ax = plt.subplots(figsize=(9, 6))

for k in ORDER:
    ax.scatter(bf[k], fm[k],
               color=col(k), s=160,
               marker="o", edgecolors="white", linewidths=0.8, zorder=5)
    ax.annotate(LABELS[k], (bf[k], fm[k]),
                textcoords="offset points",
                xytext=(6, 5), fontsize=10,
                color=col(k), fontweight="bold")

ax.set_xlabel("Best FID  (← lower is better)")
ax.set_ylabel("Mode Variance  (↑ higher is better)")
ax.set_title("Quality–Diversity Trade-off — CIFAR-10")
plt.tight_layout()
plt.show()
save_fig(fig, "09_quality_diversity_scatter")

In [ ]:
# ── Run All Plots at Once ────────────────────────────────────────────────────
def run_all():
    sm = lambda a, w=5: uniform_filter1d(np.array(a, dtype=float), size=min(w, len(a)))

    # 01 — FID CIFAR-10
    fig, ax = plt.subplots(figsize=(11, 5.5))
    for k in ORDER:
        ep, fid = get_fid(CIFAR, k)
        ax.plot(ep, fid, color=col(k), lw=1.8, marker="o", markersize=4, label=lbl(k))
    ax.set(xlabel="Epoch", ylabel="FID (↓)", title="FID Curves — CIFAR-10")
    ax.legend(fontsize=10, ncol=2)
    plt.tight_layout(); save_fig(fig, "01_fid_curves_cifar10"); plt.close()

    # 02 — FID EuroSAT
    fig, ax = plt.subplots(figsize=(11, 5.5))
    for k in ORDER:
        ep, fid = get_fid(EURO150, k)
        ax.plot(ep, fid, color=col(k), lw=1.8, marker="o", markersize=4, label=lbl(k))
    ax.set(xlabel="Epoch", ylabel="FID (↓)", title="FID Curves — EuroSAT")
    ax.legend(fontsize=10, ncol=2)
    plt.tight_layout(); save_fig(fig, "02_fid_curves_eurosat"); plt.close()

    # 03 — Loss CIFAR-10
    fig, axes = plt.subplots(2, 3, figsize=(15, 8)); axes = axes.flatten()
    for ax, k in zip(axes[:5], ORDER):
        ep = CIFAR[CIFAR["Experiment"] == k]["Epoch"].tolist()
        ax.plot(ep, sm(get_col(CIFAR, k, "G_Loss")), color=col(k), lw=2, label="Generator")
        ax.plot(ep, sm(get_col(CIFAR, k, "D_Loss")), color=col(k), lw=1.4, ls="--", alpha=0.6, label="Discriminator")
        ax.set_title(lbl(k), color=col(k), fontweight="bold")
        ax.set_xlabel("Epoch", fontsize=9); ax.set_ylabel("Loss", fontsize=9)
        ax.legend(fontsize=8)
    axes[5].set_visible(False)
    fig.suptitle("Loss Curves — CIFAR-10", fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout(); save_fig(fig, "03_loss_curves_cifar10"); plt.close()

    # 04 — Loss EuroSAT
    fig, axes = plt.subplots(2, 3, figsize=(15, 8)); axes = axes.flatten()
    for ax, k in zip(axes[:5], ORDER):
        s = EURO150[EURO150["Experiment"] == k]
        ep = s["Epoch"].tolist()
        ax.plot(ep, sm(s["G_Loss"].tolist()), color=col(k), lw=2, label="Generator")
        ax.plot(ep, sm(s["D_Loss"].tolist()), color=col(k), lw=1.4, ls="--", alpha=0.6, label="Discriminator")
        ax.set_title(lbl(k), color=col(k), fontweight="bold")
        ax.set_xlabel("Epoch", fontsize=9); ax.set_ylabel("Loss", fontsize=9)
        ax.legend(fontsize=8)
    axes[5].set_visible(False)
    fig.suptitle("Loss Curves — EuroSAT", fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout(); save_fig(fig, "04_loss_curves_eurosat"); plt.close()

    # 05 — ModeVar CIFAR-10
    fig, ax = plt.subplots(figsize=(11, 5.5))
    for k in ORDER:
        ep = CIFAR[CIFAR["Experiment"] == k]["Epoch"].tolist()
        ax.plot(ep, sm(get_col(CIFAR, k, "ModeVar"), w=7), color=col(k), lw=1.8, label=lbl(k))
    ax.set(xlabel="Epoch", ylabel="Mode Variance (↑)", title="Mode Variance — CIFAR-10")
    ax.legend(fontsize=10, ncol=2)
    plt.tight_layout(); save_fig(fig, "05_modevar_curves_cifar10"); plt.close()

    # 06 — ModeVar EuroSAT
    fig, ax = plt.subplots(figsize=(11, 5.5))
    for k in ORDER:
        s = EURO150[EURO150["Experiment"] == k]
        ax.plot(s["Epoch"].tolist(), sm(s["ModeVar"].tolist(), w=7), color=col(k), lw=1.8, label=lbl(k))
    ax.set(xlabel="Epoch", ylabel="Mode Variance (↑)", title="Mode Variance — EuroSAT")
    ax.legend(fontsize=10, ncol=2)
    plt.tight_layout(); save_fig(fig, "06_modevar_curves_eurosat"); plt.close()

    # 07 — Best FID bar
    bc = {k: min(get_fid(CIFAR,   k)[1]) for k in ORDER}
    be = {k: min(get_fid(EURO150, k)[1]) for k in ORDER}
    x = np.arange(len(ORDER)); w = 0.35
    fig, ax = plt.subplots(figsize=(12, 6))
    b1 = ax.bar(x-w/2, [bc[k] for k in ORDER], w, color=[col(k) for k in ORDER], edgecolor="white", zorder=3, label="CIFAR-10")
    b2 = ax.bar(x+w/2, [be[k] for k in ORDER], w, color=[col(k) for k in ORDER], edgecolor="white", alpha=0.55, hatch="///", zorder=3, label="EuroSAT")
    for bar in list(b1)+list(b2):
        v = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, v+0.8, f"{v:.1f}", ha="center", va="bottom", fontsize=8, color="#444")
    ax.set_xticks(x); ax.set_xticklabels([LABELS[k] for k in ORDER], rotation=15, ha="right")
    ax.set(ylabel="Best FID (↓)", title="Best FID — CIFAR-10 vs EuroSAT")
    ax.legend(fontsize=10)
    plt.tight_layout(); save_fig(fig, "07_best_fid_comparison"); plt.close()

    # 08 — ModeVar bar
    mc = {k: CIFAR[CIFAR["Experiment"]    == k]["ModeVar"].iloc[-1] for k in ORDER}
    me = {k: EURO150[EURO150["Experiment"] == k]["ModeVar"].iloc[-1] for k in ORDER}
    x = np.arange(len(ORDER)); w = 0.35
    fig, ax = plt.subplots(figsize=(12, 5.5))
    b1 = ax.bar(x-w/2, [mc[k] for k in ORDER], w, color=[col(k) for k in ORDER], edgecolor="white", zorder=3, label="CIFAR-10")
    b2 = ax.bar(x+w/2, [me[k] for k in ORDER], w, color=[col(k) for k in ORDER], edgecolor="white", alpha=0.55, hatch="///", zorder=3, label="EuroSAT")
    for bar in list(b1)+list(b2):
        v = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, v+0.002, f"{v:.3f}", ha="center", va="bottom", fontsize=7.5, color="#444")
    ax.set_xticks(x); ax.set_xticklabels([LABELS[k] for k in ORDER], rotation=15, ha="right")
    ax.set(ylabel="Mode Variance (↑)", title="Final Mode Variance — CIFAR-10 vs EuroSAT")
    ax.legend(fontsize=10)
    plt.tight_layout(); save_fig(fig, "08_modevar_bar_comparison"); plt.close()

    # 09 — Scatter
    bf = {k: min(get_fid(CIFAR, k)[1]) for k in ORDER}
    fm = {k: CIFAR[CIFAR["Experiment"] == k]["ModeVar"].iloc[-1] for k in ORDER}
    fig, ax = plt.subplots(figsize=(9, 6))
    for k in ORDER:
        ax.scatter(bf[k], fm[k], color=col(k), s=160, marker="o",
                   edgecolors="white", linewidths=0.8, zorder=5)
        ax.annotate(LABELS[k], (bf[k], fm[k]), textcoords="offset points",
                    xytext=(6, 5), fontsize=10, color=col(k), fontweight="bold")
    ax.set(xlabel="Best FID (← lower is better)",
           ylabel="Mode Variance (↑ higher is better)",
           title="Quality–Diversity Trade-off — CIFAR-10")
    plt.tight_layout(); save_fig(fig, "09_quality_diversity_scatter"); plt.close()

    print("\n✅ All 9 plots regenerated successfully.")

run_all()